# Deleting all unnecessary files


In [ ]:
import glob
import os

# Identify legacy PDF visualization exports matching project patterns
files_for_removing = glob.glob("wykres_wzrostu_*.pdf") + glob.glob("mapa_lebesgue_*.pdf")

# Purge identified files to optimize workspace storage capacity
for file in files_for_removing:
    try:
        os.remove(file)
        print(f"Removed legacy artifact: {file}")
    except Exception as e:
        print(f"Failed to remove {file} (file may be locked or permission denied): {e}")

# Approximation Node Optimizer (ANO) — Core Geometric Engine

In [ ]:
import numpy as np



# =====================================================================
# 1. GEOMETRIC OBJECTS (DOMAINS) — MULTIDIMENSIONAL TOPOLOGIES
# =====================================================================

class Domain:
    """Abstract base class establishing the interface for all geometric domains."""
    def __init__(self, name, dim=2):
        self.name = name
        self.dim = dim

    def contains(self, *args):
        """Check if given coordinates lie within the domain boundaries."""
        raise NotImplementedError

    def get_boundary(self):
        """Return coordinates to plot the 2D boundary layout."""
        return []

class Square(Domain):
    """Standard 2D Square domain bounded within [-1, 1] x [-1, 1]."""
    def __init__(self):
        super().__init__("Square", dim=2)

    def contains(self, x, y):
        return (np.abs(x) <= 1) & (np.abs(y) <= 1)

    def get_boundary(self):
        return [([-1, 1, 1, -1, -1], [-1, -1, 1, 1, -1])]

class Disk(Domain):
    """Standard 2D Unit Disk domain."""
    def __init__(self):
        super().__init__("Disk", dim=2)

    def contains(self, x, y):
        return x**2 + y**2 <= 1 + 1e-9

    def get_boundary(self):
        t = np.linspace(0, 2 * np.pi, 200)
        return [(np.cos(t), np.sin(t))]

class Sector(Domain):
    """Non-convex 3/4 Sector Disk (Unit Disk with the 3rd quadrant removed).
    Explicitly constructed via three separate, visible path segments.
    """
    def __init__(self):
        super().__init__("Sector (3/4 Disk)", dim=2)

    def contains(self, x, y):
        in_disk = x**2 + y**2 <= 1 + 1e-9
        # Third quadrant is strictly x < 0 and y < 0
        in_third_quadrant = (x < 0) & (y < 0)
        return in_disk & (~in_third_quadrant)

    def get_boundary(self):
        # 1. The circular arc segment
        t = np.linspace(-0.5 * np.pi, np.pi, 200)
        arc_x = np.cos(t)
        arc_y = np.sin(t)

        # 2. Linear segment from (-1,0) to (0,0)
        line1_x = np.linspace(-1, 0, 50)
        line1_y = np.zeros(50)

        # 3. Linear segment from (0, 0) to (0, -1)
        line2_x = np.zeros(50)
        line2_y = np.linspace(0, -1, 50)

        # Combining all segments continuously
        bx = np.concatenate([arc_x, line1_x, line2_x])
        by = np.concatenate([arc_y, line1_y, line2_y])

        return [(bx, by)]

class Triangle(Domain):
    """Centered and scaled 2D Right Triangle to match [-1, 1] bounding box."""
    def __init__(self):
        super().__init__("Triangle", dim=2)

    def contains(self, x, y):
        # Scaled to maintain legs of length 2 within [-1, 1] boundaries
        return (x >= -1) & (y >= -1) & (x + y <= 0)

    def get_boundary(self):
        return [([-1, 1, -1, -1], [-1, -1, 1, -1])]

class DoubleSquare(Domain):
    """Two distinct square domains connected via a narrow bridge coordinate system."""
    def __init__(self):
        super().__init__("Double Square", dim=2)

    def contains(self, x, y):
        sq1 = (x >= -1.5) & (x <= -0.5) & (y >= -0.5) & (y <= 0.5)
        sq2 = (x >= 0.5) & (x <= 1.5) & (y >= -0.5) & (y <= 0.5)
        bridge = (x >= -0.5) & (x <= 0.5) & (np.abs(y) <= 0.1)
        return sq1 | sq2 | bridge

    def get_boundary(self):
        return [([-1.5, -0.5, -0.5, 0.5, 0.5, 1.5, 1.5, 0.5, 0.5, -0.5, -0.5, -1.5, -1.5],
                 [-0.5, -0.5, -0.1, -0.1, -0.5, -0.5, 0.5, 0.5, 0.1, 0.1, 0.5, 0.5, -0.5])]

class SquareDumbbell(Domain):
    """Square dumbbell configuration."""
    def __init__(self):
        super().__init__("Square Dumbbell", dim=2)

    def contains(self, x, y):
        sq1 = (x >= -1.5) & (x <= -0.5) & (y >= -0.5) & (y <= 0.5)
        sq2 = (x >= 0.5) & (x <= 1.5) & (y >= -0.5) & (y <= 0.5)
        bridge = (x >= -0.5) & (x <= 0.5) & (np.abs(y) <= 0.1)
        return sq1 | sq2 | bridge

    def get_boundary(self):
        return [([-1.5, -0.5, -0.5, 0.5, 0.5, 1.5, 1.5, 0.5, 0.5, -0.5, -0.5, -1.5, -1.5],
                 [-0.5, -0.5, -0.1, -0.1, -0.5, -0.5, 0.5, 0.5, 0.1, 0.1, 0.5, 0.5, -0.5])]

class RoundDumbbell(Domain):
    """Round dumbbell with precise continuous boundary tracking."""
    def __init__(self, r=0.5, dist=1.0, bar_w=0.1):
        super().__init__("Round Dumbbell", dim=2)
        self.r = r
        self.dist = dist
        self.bar_w = bar_w

    def contains(self, x, y):
        disk1 = (x + self.dist)**2 + y**2 <= self.r**2 + 1e-9
        disk2 = (x - self.dist)**2 + y**2 <= self.r**2 + 1e-9
        bridge = (x >= -self.dist) & (x <= self.dist) & (np.abs(y) <= self.bar_w)
        return disk1 | disk2 | bridge

    def get_boundary(self):
        # Precise angle intersection between the bridge and disk circumference
        alpha = np.arcsin(self.bar_w / self.r)

        # 1. Left arc (clockwise traversal from top-bridge to bottom-bridge)
        t_left = np.linspace(alpha, 2 * np.pi - alpha, 100)
        left_x = -self.dist + self.r * np.cos(t_left)
        left_y = self.r * np.sin(t_left)

        # 2. Right arc (clockwise traversal from bottom-bridge to top-bridge)
        t_right = np.linspace(np.pi + alpha, 3 * np.pi - alpha, 100)
        right_x = self.dist + self.r * np.cos(t_right)
        right_y = self.r * np.sin(t_right)

        # Assembly of the continuous path without discontinuity jumps
        bx = np.concatenate([left_x, right_x, [left_x[0]]])
        by = np.concatenate([left_y, right_y, [left_y[0]]])
        return [(bx, by)]

class Annulus(Domain):
    """2D Annulus domain bounded by an inner radius (r_in) and outer radius (r_out)."""
    def __init__(self, r_in=0.4, r_out=1.0):
        super().__init__("Annulus", dim=2)
        self.r_in, self.r_out = r_in, r_out

    def contains(self, x, y):
        r2 = x**2 + y**2
        return (r2 >= self.r_in**2) & (r2 <= self.r_out**2)

    def get_boundary(self):
        t = np.linspace(0, 2 * np.pi, 200)
        return [(self.r_out * np.cos(t), self.r_out * np.sin(t)),
                (self.r_in * np.cos(t), self.r_in * np.sin(t))]

class Rhombus(Domain):
    """Standard 2D Rhombus (Diamond) domain layout."""
    def __init__(self):
        super().__init__("Rhombus", dim=2)

    def contains(self, x, y):
        return np.abs(x) + np.abs(y) <= 1

    def get_boundary(self):
        return [([1, 0, -1, 0, 1], [0, 1, 0, -1, 0])]

class EquilateralTriangle(Domain):
    """Centered Equilateral Triangle scaled to fit the [-1, 1] bounding box."""
    def __init__(self):
        super().__init__("Equilateral Triangle", dim=2)

    def contains(self, x, y):
        # Scaled linear boundary conditions for triangle inscribed in R=1 circle
        cond1 = y >= -0.5 - 1e-9
        cond2 = y <= -np.sqrt(3) * x + 1.0 + 1e-9
        cond3 = y <= np.sqrt(3) * x + 1.0 + 1e-9
        return cond1 & cond2 & cond3

    def get_boundary(self):
        # Setting accurate vertex coordinates
        x_left = -np.sqrt(3) / 2
        x_right = np.sqrt(3) / 2
        y_bottom = -0.5
        y_top = 1.0

        # Closing loop path to the starting vertex
        bx = [x_left, x_right, 0.0, x_left]
        by = [y_bottom, y_bottom, y_top, y_bottom]
        return [(np.array(bx), np.array(by))]

class DisjointSquares(Domain):
    """Two completely separate, disconnected square spaces to analyze nodal partitioning."""
    def __init__(self):
        super().__init__("Disjoint Squares", dim=2)

    def contains(self, x, y):
        sq1 = (np.abs(x + 1) <= 0.4) & (np.abs(y) <= 0.4)
        sq2 = (np.abs(x - 1) <= 0.4) & (np.abs(y) <= 0.4)
        return sq1 | sq2

    def get_boundary(self):
        return [([-1.4, -0.6, -0.6, -1.4, -1.4], [-0.4, -0.4, 0.4, 0.4, -0.4]),
                ([0.6, 1.4, 1.4, 0.6, 0.6], [-0.4, -0.4, 0.4, 0.4, -0.4])]

class Interval1D(Domain):
    """Standard 1D canonical interval domain mapped to [-1, 1]."""
    def __init__(self):
        super().__init__("Interval 1D", dim=1)

    def contains(self, x):
        return np.abs(x) <= 1

    def get_boundary(self):
        return [([-1, 1], [0, 0])]

# =====================================================================
# 3D SPATIAL MANIFOLDS
# =====================================================================

class Sphere3D(Domain):
    """3D Solid Unit Sphere volume domain."""
    def __init__(self):
        super().__init__("Sphere 3D", dim=3)

    def contains(self, x, y, z):
        return x**2 + y**2 + z**2 <= 1 + 1e-9

    def plot_surface(self, ax):
        """Generate wireframe mesh surface rendering for matplotlib 3D visualizers."""
        u, v = np.mgrid[0:2*np.pi:30j, 0:np.pi:15j]
        x = np.cos(u) * np.sin(v)
        y = np.sin(u) * np.sin(v)
        z = np.cos(v)
        ax.plot_wireframe(x, y, z, color="green", alpha=0.3, linewidth=0.5)

class Cylinder3D(Domain):
    """3D Solid Cylinder volume configuration with custom radius and height."""
    def __init__(self, r=1.0, h=2.0):
        super().__init__("Cylinder 3D", dim=3)
        self.r, self.h = r, h

    def contains(self, x, y, z):
        return (x**2 + y**2 <= self.r**2) & (np.abs(z) <= self.h / 2)

    def plot_surface(self, ax):
        """Generate side-wall wireframe mesh surface visualization."""
        z_steps = np.linspace(-self.h / 2, self.h / 2, 20)
        theta = np.linspace(0, 2 * np.pi, 40)
        theta_grid, z_grid = np.meshgrid(theta, z_steps)
        x_side, y_side = self.r * np.cos(theta_grid), self.r * np.sin(theta_grid)
        ax.plot_wireframe(x_side, y_side, z_grid, color='gray', alpha=0.2, lw=0.5)
        ax.set_box_aspect([1, 1, self.h / (2 * self.r)])

class Torus3D(Domain):
    """3D Solid Torus volume domain based on major radius (R) and minor tube radius (r)."""
    def __init__(self, R=1.0, r=0.3):
        super().__init__("Torus 3D", dim=3)
        self.R, self.r = R, r

    def contains(self, x, y, z):
        rho = np.sqrt(x**2 + y**2)
        return (rho - self.R)**2 + z**2 <= self.r**2

    def plot_surface(self, ax):
        """Generate torus wireframe mesh rendering."""
        u = np.linspace(0, 2 * np.pi, 40)
        v = np.linspace(0, 2 * np.pi, 20)
        U, V = np.meshgrid(u, v)
        X = (self.R + self.r * np.cos(V)) * np.cos(U)
        Y = (self.R + self.r * np.cos(V)) * np.sin(U)
        Z = self.r * np.sin(V)
        ax.plot_wireframe(X, Y, Z, color='black', alpha=0.15, linewidth=0.5)
        ax.set_box_aspect([1, 1, self.r / self.R])


# Mathematical Engine & Nodal Generation Solvers

In [ ]:
import numpy as np
from scipy.linalg import qr

# =====================================================================
# 2. MATHEMATICAL ENGINE (MULTIDIMENSIONAL CHEBYSHEV POLYNOMIALS)
# =====================================================================

def halton_sequence(size, dim):
    """Generates a strictly deterministic, evenly distributed quasi-random
    sequence for dimensions 1, 2, and 3 to ensure reproducible space filling.
    """
    seq = np.zeros((size, dim))
    primes = [2, 3, 5]

    for d in range(dim):
        base = primes[d]
        n = np.arange(1, size + 1)
        fraction = np.zeros(size)
        denominator = base
        while np.any(n > 0):
            fraction += (n % base) / denominator
            n //= base
            denominator *= base
        seq[:, d] = fraction

    return seq


def get_universal_candidates(domain, n_candidates=4000):
    """Constructs a stable, repeatable candidate mesh by combining
    exact continuous boundary paths with a deterministic Halton core.
    """
    cand = []

    # 1. Capture exact boundary manifolds for 1D and 2D spaces
    if domain.dim < 3:
        boundary_info = domain.get_boundary()
        if boundary_info:
            for seg in boundary_info:
                bx, by = np.array(seg[0]), np.array(seg[1])
                if domain.dim == 1:
                    pts_boundary = bx.reshape(-1, 1)
                else:
                    pts_boundary = np.vstack([bx, by]).T
                cand.extend(pts_boundary)

    # 2. Populate domain interior using deterministic low-discrepancy points
    bbox_limit = 2.2 if "Square" in domain.name or "Double" in domain.name else 1.5
    raw_size = n_candidates * 5
    raw_pts = halton_sequence(raw_size, domain.dim)

    # Map from canonical [0, 1] space to bounding box limits
    scaled_pts = -bbox_limit + 2 * bbox_limit * raw_pts

    # Filter nodes through geometric indicator condition functions
    mask = domain.contains(*[scaled_pts[:, i] for i in range(domain.dim)])
    filtered_pts = scaled_pts[mask]

    cand.extend(filtered_pts)
    return np.array(cand)[:n_candidates]


def get_vander(nodes, degree, domain):
    """Constructs a stable multidimensional Vandermonde matrix using
    normalized Chebyshev polynomials mapped strictly to the [-1.0, 1.0] interval
    based on the domain bounding box to guarantee unique polynomial basis.
    """
    n_nodes = nodes.shape[0]
    dim = nodes.shape[1]

    nodes_norm = nodes.copy()

    # Normalize coordinates using global domain geometry bounds
    # to guarantee identical algebraic basis for test and interpolation sets.
    if "Square" in domain.name or "Double" in domain.name:
        bbox_limit = 2.2
    else:
        bbox_limit = 1.5

    for d in range(dim):
        nodes_norm[:, d] = nodes[:, d] / bbox_limit

    def cheb_val(coords, deg):
        return np.cos(deg * np.arccos(np.clip(coords, -1.0, 1.0)))

    if dim == 1:
        x = nodes_norm[:, 0]
        V = np.zeros((n_nodes, degree + 1))
        for i in range(degree + 1):
            V[:, i] = cheb_val(x, i)
        return V

    elif dim == 2:
        x, y = nodes_norm[:, 0], nodes_norm[:, 1]
        columns = []
        for i in range(degree + 1):
            for j in range(degree + 1 - i):
                columns.append(cheb_val(x, i) * cheb_val(y, j))
        return np.column_stack(columns)

    else:  # 3D
        x, y, z = nodes_norm[:, 0], nodes_norm[:, 1], nodes_norm[:, 2]
        columns = []
        for i in range(degree + 1):
            for j in range(degree + 1 - i):
                for k in range(degree + 1 - i - j):
                    columns.append(cheb_val(x, i) * cheb_val(y, j) * cheb_val(z, k))
        return np.column_stack(columns)


def calculate_lebesgue(nodes, domain, degree, res=100):
    """Computes an empirical approximation of the Lebesgue constant by
    evaluating the operator norm across an adaptive bounding-box grid.
    """
    x_min, x_max = nodes[:, 0].min(), nodes[:, 0].max()

    if domain.dim == 1:
        test_pts = np.linspace(x_min, x_max, 1500).reshape(-1, 1)
        mask = np.ones(len(test_pts), dtype=bool)
        gx, gy = test_pts, None
    elif domain.dim == 2:
        y_min, y_max = nodes[:, 1].min(), nodes[:, 1].max()

        # Adaptive test mesh strictly matched to node configuration limits
        _tx = np.linspace(x_min, x_max, res)
        _ty = np.linspace(y_min, y_max, res)

        gx, gy = np.meshgrid(_tx, _ty)
        test_pts = np.vstack([gx.ravel(), gy.ravel()]).T
        mask = domain.contains(test_pts[:, 0], test_pts[:, 1])
    else:  # 3D Manifolds
        # Increased grid mesh resolution for precise volume tracking
        _t = np.linspace(-1.5, 1.5, 45)
        gx, gy, gz = np.meshgrid(_t, _t, _t)
        test_pts = np.vstack([gx.ravel(), gy.ravel(), gz.ravel()]).T
        mask = domain.contains(test_pts[:, 0], test_pts[:, 1], test_pts[:, 2])

    test_in = test_pts[mask]
    if len(test_in) == 0:
        return 0.0, (gx, gy, np.array([]), mask, np.zeros(domain.dim))

    V_nodes = get_vander(nodes, degree, domain)
    V_test = get_vander(test_in, degree, domain)

    L_mat = V_test @ np.linalg.pinv(V_nodes, rcond=1e-15)
    l_vals = np.sum(np.abs(L_mat), axis=1)

    l_max = np.max(l_vals)
    max_point = test_in[np.argmax(l_vals)]

    return l_max, (gx, gy, l_vals, mask, max_point)


# =====================================================================
# 3. NODAL GENERATION SOLVERS (FEKETE, LEJA, PADUA, BOS)
# =====================================================================

def get_fekete_discrete(domain, degree, n_candidates=6000):
    """Generates Approximate Fekete Points using QR-preconditioning
    based on row-pivoting techniques for stable maximum determinant selection.
    """
    if degree > 40:  # Hardware cap optimization
        import warnings
        warnings.warn("Polynomial degree capped at 40 to ensure hardware execution stability.", RuntimeWarning)
        degree = 40

    cand = get_universal_candidates(domain, n_candidates)
    V_huge = get_vander(cand, degree, domain)

    # Condition stabilization using reduced QR factorization
    Q_stable, _ = np.linalg.qr(V_huge, mode='reduced')

    # Linear row selection via pivoting on orthogonal column spaces
    _, _, p = qr(Q_stable.T, pivoting=True)

    return cand[p[:V_huge.shape[1]]]


def get_nodes(method, domain, degree):
    """Central interface module for distributing nodal generation requests to specific
    discrete mathematical solvers based on requested dimensions and geometric metrics.
    """
    if domain.dim == 1:
        n_pts = degree + 1
    elif domain.dim == 2:
        n_pts = (degree + 1) * (degree + 2) // 2
    else:
        n_pts = (degree + 1) * (degree + 2) * (degree + 3) // 6

    # --- Domain-agnostic numerical methods ---
    if method == "fekete":
        return get_fekete_discrete(domain, degree)

    if method == "leja" and domain.dim >= 2:
        n_candidates = 6000
        cand = get_universal_candidates(domain, n_candidates)
        V_cand = get_vander(cand, degree, domain)

        selected_indices = []
        A = V_cand.copy().astype(float)

        for step in range(n_pts):
            idx = np.argmax(np.abs(A[step:, step])) + step
            selected_indices.append(idx)

            A[[step, idx]] = A[[idx, step]]
            cand[[step, idx]] = cand[[idx, step]]

            pivot = A[step, step]
            if np.abs(pivot) > 1e-14:
                factors = A[step + 1:, step] / pivot
                A[step + 1:, step:] -= np.outer(factors, A[step, step:])

        return cand[:n_pts]

    # --- 1-Dimensional configurations (1D) ---
    if domain.dim == 1:
        if method == "chebyshev":
            return np.cos((2 * np.arange(1, degree + 2) - 1) * np.pi / (2 * (degree + 1))).reshape(-1, 1)
        if method == "uniform":
            return np.linspace(-1, 1, n_pts).reshape(-1, 1)
        if method == "leja":
            cand = np.linspace(-1, 1, 2000).reshape(-1, 1)
            pts = [1.0]
            for _ in range(n_pts - 1):
                dists = np.abs(cand - np.array(pts).reshape(1, -1))
                potential = np.sum(np.log(dists + 1e-15), axis=1)
                pts.append(cand[np.argmax(potential), 0])
            return np.array(pts).reshape(-1, 1)

    # --- Analytical structural topologies (2D) ---
    if domain.dim == 2:
        if method == "chebyshev_tensor" and "Square" in domain.name:
            t = np.cos((2 * np.arange(1, degree + 1) - 1) * np.pi / (2 * degree))
            gx, gy = np.meshgrid(t, t)
            return np.vstack([gx.ravel(), gy.ravel()]).T
        if method == "padua" and "Square" in domain.name:
            pts = []
            for i in range(degree + 1):
                for j in range(degree + 2):
                    if (i + j) % 2 == 1:
                        x_val = np.cos(j * np.pi / (degree + 1))
                        y_val = np.cos(i * np.pi / degree)
                        pts.append((x_val, y_val))
            return np.array(pts)
        if method == "bos" and "Disk" in domain.name:
            pts = []
            for k in range(degree // 2 + 1):
                r = np.cos(k * np.pi / degree)
                m = 2 * degree + 1 - 4 * k
                if m > 0:
                    for t in np.linspace(0, 2 * np.pi, m, endpoint=False):
                        pts.append((r * np.cos(t), r * np.sin(t)))
            return np.array(pts)

    # --- Bounding box uniform distribution baseline ---
    if method == "uniform":
        pts = []
        limit = 2.2 if "Square" in domain.name or "Double" in domain.name else 1.5
        while len(pts) < n_pts:
            p = np.random.uniform(-limit, limit, domain.dim)
            if domain.contains(*p):
                pts.append(p)
        return np.array(pts)

    return None

# Visualization Suite, Dynamic Interactive UI & LaTeX Exporter

In [ ]:
# import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from mpl_toolkits.mplot3d import Axes3D

# =====================================================================
# 4. EXPERIMENT INTERFACE AND PROGRESS TRACKING
# =====================================================================

def run_experiment(domain_obj, method, degree, res=150):
    """Executes a complete nodal generation experiment, calculates interpolation
    stability via the Lebesgue constant, and renders structural plots.
    """
    nodes = get_nodes(method, domain_obj, degree)
    if nodes is None:
        print(f"❌ Method '{method}' is incompatible with geometry: {domain_obj.name}")
        return None

    l_max, data = calculate_lebesgue(nodes, domain_obj, degree, res)

    print(f"✅ Experiment Successful: {domain_obj.name} | Method: {method} | Nodes N={len(nodes)} | L_max={l_max:.2f}")

    if domain_obj.dim == 1:
        plt.figure(figsize=(10, 3.5))
        tx = np.linspace(-1, 1, 1000).reshape(-1, 1)
        _, d_1d = calculate_lebesgue(nodes, domain_obj, degree, res)
        plt.plot(tx, d_1d[2], 'b-', lw=1.5, label=r'Lebesgue Function $\Lambda_n(x)$')
        plt.scatter(nodes, np.zeros_like(nodes), c='red', s=40, label='Nodes', zorder=5)
        plt.scatter(d_1d[4], l_max, c='cyan', marker='X', s=100, label=f'Max Peak: {l_max:.2f}', zorder=10)
        plt.title(f"1D Stability Analysis: {method} on {domain_obj.name} (Degree {degree})")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()

    elif domain_obj.dim == 2:
        plt.figure(figsize=(6, 5.5))
        for bx, by in domain_obj.get_boundary():
            plt.plot(bx, by, 'k-', lw=2)
        plt.scatter(nodes[:, 0], nodes[:, 1], c='red', s=25, edgecolors='k', label='Nodes', zorder=5)
        plt.scatter(data[4][0], data[4][1], c='blue', s=150, marker='*', edgecolors='w', label=f'L_max: {l_max:.2f}', zorder=10)
        plt.title(f"Nodal Layout: {method} on {domain_obj.name} (Degree {degree})")

        # Adaptive axis framing based on actual distribution bounds
        plt.xlim(nodes[:, 0].min() - 0.2, nodes[:, 0].max() + 0.2)
        plt.ylim(nodes[:, 1].min() - 0.2, nodes[:, 1].max() + 0.2)

        plt.gca().set_aspect('equal')
        plt.grid(True, alpha=0.2)
        plt.legend()
        plt.show()

    return nodes


# =====================================================================
# 5. HIGH-FIDELITY ACADEMIC VISUALIZATIONS (CONTOURS & 3D RELIEFS)
# =====================================================================

def plot_lebesgue_analysis_2d(nodes, domain, degree, res=500):
    """Generates a high-resolution 2D Lebesgue function contour map
    using a dense grid to eliminate pixelation artifacts on curved boundaries.
    """
    l_max, data = calculate_lebesgue(nodes, domain, degree, res=res)
    gx, gy, l_vals, mask, max_point = data

    # Initialize matrix map Z: values inside domain, NaN outside
    Z = np.full(gx.shape, np.nan)
    Z.ravel()[mask] = l_vals

    fig, ax = plt.subplots(figsize=(8.5, 7))

    cp = ax.contourf(gx, gy, Z, levels=60, cmap='viridis', alpha=0.85, corner_mask=True)
    contours = ax.contour(gx, gy, Z, levels=14, colors='white', linewidths=0.5, alpha=0.4, corner_mask=True)
    ax.clabel(contours, inline=True, fontsize=8)

    fig.colorbar(cp, ax=ax, label=r'$\Lambda_n(x, y)$ Operator Norm')

    # Continuous boundary tracking lines overlay
    for bx, by in domain.get_boundary():
        ax.plot(bx, by, 'k-', lw=2.5, zorder=8)

    # Distribution nodes plot
    ax.scatter(nodes[:, 0], nodes[:, 1], c='red', s=35, edgecolors='k', label=f'Nodes ($N={len(nodes)}$)', zorder=10)

    # Global operator maximum peak coordinate flag
    ax.scatter(max_point[0], max_point[1], c='cyan', s=180, marker='*', edgecolors='k', label=f'Max Peak: {l_max:.2f}', zorder=12)

    ax.set_xlim(nodes[:, 0].min() - 0.05, nodes[:, 0].max() + 0.05)
    ax.set_ylim(nodes[:, 1].min() - 0.05, nodes[:, 1].max() + 0.05)

    ax.set_title(f"Lebesgue Map: {domain.name} (Degree $n={degree}$)")
    ax.set_aspect('equal')
    ax.legend()
    plt.show()

def plot_lebesgue_surface_3d(nodes, domain, degree, res=180):
    """Renders a clean 3D elevation surface representing the geometric topography
    of the Lebesgue error function, strictly masked by the analytical domain geometry.
    """
    l_max, data = calculate_lebesgue(nodes, domain, degree, res=res)
    gx, gy, l_vals, mask, max_point = data

    Z = np.full(gx.shape, np.nan)
    Z.ravel()[mask] = l_vals

    fig = plt.figure(figsize=(11, 7))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(gx, gy, Z, cmap='plasma', alpha=0.85, linewidth=0, antialiased=True)

    offset = np.nanmin(l_vals) - 1.0
    ax.contourf(gx, gy, Z, zdir='z', offset=offset, cmap='plasma', alpha=0.2)

    for bx, by in domain.get_boundary():
        ax.plot(bx, by, offset, 'k-', lw=2, zorder=10)
    ax.scatter(nodes[:, 0], nodes[:, 1], offset, c='red', s=20, edgecolors='k', zorder=12)
    ax.scatter(max_point[0], max_point[1], l_max, c='yellow', s=150, marker='*', edgecolors='black', zorder=20)

    ax.set_xlim(nodes[:, 0].min() - 0.05, nodes[:, 0].max() + 0.05)
    ax.set_ylim(nodes[:, 1].min() - 0.05, nodes[:, 1].max() + 0.05)

    ax.set_title(f"3D Topographic Surface of the Lebesgue Function\n$L_{{max}} = {l_max:.3f}$")
    ax.view_init(elev=28, azim=35)
    fig.colorbar(surf, shrink=0.5, aspect=10)
    plt.show()


# =====================================================================
# 6. INTERACTIVE LABORATORY AND DYNAMIC COORD-MANIPULATORS
# =====================================================================

def final_interactive_lab(domain, method="fekete", degree=3):
    """Launches an interactive GUI framework enabling users to dynamically drag
    individual nodes and observe real-time destabilization shifts in Lebesgue parameters.
    """
    nodes = get_nodes(method, domain, degree)
    step = 0.04
    out = widgets.Output()

    def redraw(target_idx):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(6, 6))
            for bx, by in domain.get_boundary():
                ax.plot(bx, by, 'k-', lw=2)

            l_max, data = calculate_lebesgue(nodes, domain, degree, res=80)
            _, _, _, _, max_point = data

            ax.scatter(nodes[:, 0], nodes[:, 1], c='red', s=50, edgecolors='k', alpha=0.6, label='Nodes')
            ax.scatter(max_point[0], max_point[1], c='blue', s=180, marker='*', edgecolors='w', zorder=10, label='Lebesgue Peak')
            ax.scatter(nodes[target_idx, 0], nodes[target_idx, 1], c='yellow', s=120, edgecolors='r', zorder=11, label='Active Selected Node')
            ax.set_title(f"Dynamic Lab | Real-Time L_max: {l_max:.4f}")

            ax.set_xlim(nodes[:, 0].min() - 0.2, nodes[:, 0].max() + 0.2)
            ax.set_ylim(nodes[:, 1].min() - 0.2, nodes[:, 1].max() + 0.2)

            ax.set_aspect('equal')
            ax.legend()
            plt.show()

    point_selector = widgets.IntSlider(value=0, min=0, max=len(nodes)-1, description='Node Index:')
    btn_up = widgets.Button(description='▲', layout=widgets.Layout(width='40px'))
    btn_down = widgets.Button(description='▼', layout=widgets.Layout(width='40px'))
    btn_left = widgets.Button(description='◀', layout=widgets.Layout(width='40px'))
    btn_right = widgets.Button(description='▶', layout=widgets.Layout(width='40px'))

    def move(b):
        idx = point_selector.value
        new_pos = nodes[idx].copy()
        if b.description == '▲': new_pos[1] += step
        if b.description == '▼': new_pos[1] -= step
        if b.description == '◀': new_pos[0] -= step
        if b.description == '▶': new_pos[0] += step
        if domain.contains(np.array([new_pos[0]]), np.array([new_pos[1]])):
            nodes[idx] = new_pos
            redraw(idx)

    for btn in [btn_up, btn_down, btn_left, btn_right]:
        btn.on_click(move)
    point_selector.observe(lambda x: redraw(point_selector.value), names='value')

    ctrls = widgets.VBox([
        point_selector,
        widgets.HBox([widgets.Label(layout={"width":"40px"}), btn_up]),
        widgets.HBox([btn_left, btn_down, btn_right])
    ])
    display(widgets.HBox([ctrls, out]))
    redraw(0)


def interactive_3d_slice(nodes, method_name, domain):
    """Executes volumetric spatial parsing for 3D topologies, projecting a narrow horizontal
    coordinate window into a standalone 2D cross-section graph plane.
    """
    margin = 1.1
    x_lim = [nodes[:, 0].min() * margin, nodes[:, 0].max() * margin]
    y_lim = [nodes[:, 1].min() * margin, nodes[:, 1].max() * margin]
    z_min, z_max = nodes[:, 2].min(), nodes[:, 2].max()

    z_slider = widgets.FloatSlider(value=(z_min+z_max)/2, min=z_min, max=z_max, step=(z_max-z_min)/30, description='Z Slice Level:')
    width_slider = widgets.FloatSlider(value=(z_max-z_min)*0.15, min=0.01, max=(z_max-z_min)*0.4, description='Window Thickness:')
    out = widgets.Output()

    def update_slice(change):
        zv, w = z_slider.value, width_slider.value
        mask = (nodes[:, 2] >= zv - w) & (nodes[:, 2] <= zv + w)
        slice_pts = nodes[mask]
        with out:
            clear_output(wait=True)
            fig = plt.figure(figsize=(11, 4.5))
            ax1 = fig.add_subplot(121, projection='3d')
            ax1.scatter(nodes[:,0], nodes[:,1], nodes[:,2], c='gray', alpha=0.03, s=4)
            if hasattr(domain, 'plot_surface'):
                domain.plot_surface(ax1)
            if len(slice_pts) > 0:
                ax1.scatter(slice_pts[:,0], slice_pts[:,1], slice_pts[:,2], c='red', s=25)

            ax2 = fig.add_subplot(122)
            ax2.scatter(slice_pts[:, 0], slice_pts[:, 1], c='blue', s=35, edgecolors='k')
            ax2.set_xlim(x_lim)
            ax2.set_ylim(y_lim)
            ax2.set_aspect('equal')
            ax2.grid(True)
            ax2.set_title(f"2D Intersection Slice Plane at Z={zv:.2f} (Extracted Nodes: {len(slice_pts)})")
            plt.tight_layout()
            plt.show()

    z_slider.observe(update_slice, 'value')
    width_slider.observe(update_slice, 'value')
    display(widgets.VBox([z_slider, width_slider, out]))
    update_slice(None)

# Executive Research Sandbox & Verification Experiments

In [ ]:
# =====================================================================
# 7. EXECUTION SANDBOX: DEMONSTRATION AND VALIDATION RUNS
# =====================================================================

print("--- Step 1: Initializing Geometric Domains ---")
square_domain = Square()
sector_disk = Sector()
torus_3d = Torus3D()

print("\n--- Step 2: Running Standard 2D Numerical Experiments ---")
# Experiment A: Approximate Fekete Points on a complex non-convex 3/4 Sector Disk
# Generates nodes, calculates Lebesgue constant, and renders diagnostic plots
nodes_fekete = run_experiment(sector_disk, method="fekete", degree=4, res=80)

# Experiment B: Multidimensional Leja Sequences on a standard Unit Disk domain
nodes_leja = run_experiment(Disk(), method="leja", degree=15, res=80)


print("\n--- Step 3: Generating High-Resolution Academic Graphs ---")
if nodes_fekete is not None:
    # Render the 2D Isoline Contour Heatmap
    plot_lebesgue_analysis_2d(nodes_fekete, sector_disk, degree=4, res=100)

    # Render the 3D Lebesgue Topographic Surface Relief
    plot_lebesgue_surface_3d(nodes_fekete, sector_disk, degree=4, res=100)


print("\n--- Step 4: Activating Interactive Laboratory Module ---")
# Launches the GUI framework to manipulate node coordinates manually
final_interactive_lab(square_domain, method="fekete", degree=3)


print("\n--- Step 5: Testing Volumetric 3D Manifold Partitioning ---")
# Generates 3D points inside a Solid Torus and runs the dynamic Z-slice tool
nodes_torus = get_nodes("leja", torus_3d, degree=4)
if nodes_torus is not None:
    interactive_3d_slice(nodes_torus, "leja", torus_3d)

# Function for tables and plots of growth


In [ ]:
def make_table_and_plot_with_map(domain, method, degree_list, target_deg=9, res=80):
    import matplotlib.pyplot as plt
    import numpy as np

    degrees = []
    lebesgue_constants = []
    rows_data = []
    nodes_target = None

    # Generate an OS-safe identifier string for storage paths
    safe_domain_name = domain.name.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('/', '')

    print(f"=== ANALYSIS LOG: {domain.name} ({method.upper()}) ===")
    print("Degree n   |  Lebesgue Constant   |  Maximum Peak Coordinates (x, y)")
    print("-" * 65)

    for deg in degree_list:
        nodes = get_nodes(method, domain, deg)
        if nodes is None:
            continue

        l_max, data = calculate_lebesgue(nodes, domain, deg, res=res)
        max_pt = data[4]
        pt_str = f"({max_pt[0]:.3f}, {max_pt[1]:.3f})" if len(max_pt) >= 2 else "N/A"

        print(f"n = {deg:<6} |  L_max = {l_max:<12.4f} |  Peak = {pt_str}")

        degrees.append(deg)
        lebesgue_constants.append(l_max)
        rows_data.append((deg, l_max, pt_str))

        if deg == target_deg:
            nodes_target = nodes.copy()

    print("-" * 65)

    # Automated LaTeX Tabular Environment Block Generation
    print("\n% LATEX TABULAR SOURCE BLOCK (CLIPBOARD COPY):")
    print(r"\begin{table}[h!]\centering")
    print(r"\begin{tabular}{|c|c|c|}\hline")
    print(r" Stopień $n$ & Stała Lebesgue'a $\Lambda_n$ & Punkt krytyczny $(x_{max}, y_{max})$ \\ \hline")
    for deg, l_max, pt_str in rows_data:
        print(f" {deg} & {l_max:.4f} & {pt_str} \\\\ \\hline")
    print(r"\end{tabular}")
    print(f"\\caption{{Wartości stałej Lebesgue'a dla obszaru {domain.name} i metody {method}.}}")
    print(f"\\label{{tab:lebesgue_{safe_domain_name}_{method}}}")
    print(r"\end{table}")

    # 1. Generate and save the operator growth tracking curve
    plt.figure(figsize=(6, 4))
    plt.plot(degrees, lebesgue_constants, 'o-', color='red', linewidth=2)
    plt.xlabel('Stopień n')
    plt.ylabel('L_max')
    plt.grid(True)
    plt.title(f"Wzrost L_max: {domain.name} ({method})")

    wzrost_filename = f"wykres_wzrostu_{safe_domain_name}_{method}.pdf"
    plt.savefig(wzrost_filename, bbox_inches='tight')
    plt.show()
    plt.close()

    # 2. Extract high-resolution contour field configuration
    if nodes_target is not None:
        mapa_filename = f"mapa_lebesgue_n{target_deg}_{safe_domain_name}_{method}.pdf"
        original_show = plt.show

        # Interceptor closure: guarantees persistent file storage prior to environment rendering
        def custom_show(*args, **kwargs):
            plt.savefig(mapa_filename, bbox_inches='tight')
            print(f"📸 [Interceptor] Isoline map successfully captured and stored to: {mapa_filename}")
            original_show(*args, **kwargs)

        plt.show = custom_show

        try:
            plot_lebesgue_analysis_2d(nodes_target, domain, degree=target_deg, res=res)
        finally:
            # Revert state vector to prevent environment context leaking
            plt.show = original_show
            plt.close('all')
    else:
        print(f"\n⚠️ Target map generation bypassed for n={target_deg} (specified degree index absent from runtime execution matrix).")

### Numerical Experiments on a 2D Disk Domain

*Note: The computational pipelines, parameter matrices ($n \in [3, 15]$), and visualization functions for all alternative 2D geometric domains are implemented analogously.*

In [ ]:
# Define uniform polynomial execution limits to maintain stability profiles
test_degrees = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
chosen_map_deg = 9

disk = Disk()

print("🧪 RUNNING EXPERIMENT 1: Bos Nodes on a Unit Disk")
make_table_and_plot_with_map(disk, "bos", test_degrees, target_deg=chosen_map_deg, res=200)

print("\n🧪 RUNNING EXPERIMENT 2: Approximate Fekete Points on a Unit Disk")
make_table_and_plot_with_map(disk, "fekete", test_degrees, target_deg=chosen_map_deg, res=200)

print("\n🧪 RUNNING EXPERIMENT 3: Discrete Leja Sequences on a Unit Disk")
make_table_and_plot_with_map(disk, "leja", test_degrees, target_deg=chosen_map_deg, res=200)

In [ ]:
square = Square()

print("🧪 RUNNING EXPERIMENT: Approximate Fekete Points on a Standard Square")
make_table_and_plot_with_map(square, "fekete", test_degrees, target_deg=chosen_map_deg, res=150)

print("\n🧪 RUNNING EXPERIMENT: Discrete Leja Sequences on a Standard Square")
make_table_and_plot_with_map(square, "leja", test_degrees, target_deg=chosen_map_deg, res=150)

print("\n🧪 RUNNING EXPERIMENT: Analytical Padua Points on a Standard Square")
make_table_and_plot_with_map(square, "padua", test_degrees, target_deg=chosen_map_deg, res=150)

In [ ]:
sector = Sector()

print("🧪 RUNNING EXPERIMENT: Approximate Fekete Points on a 3/4 Circular Sector")
make_table_and_plot_with_map(sector, "fekete", test_degrees, target_deg=chosen_map_deg, res=200)

print("\n🧪 RUNNING EXPERIMENT: Discrete Leja Sequences on a 3/4 Circular Sector")
make_table_and_plot_with_map(sector, "leja", test_degrees, target_deg=chosen_map_deg, res=200)

In [ ]:
sq_dumbbell = DoubleSquare()

print("🧪 RUNNING EXPERIMENT: Approximate Fekete Points on a Square Dumbbell Manifold")
make_table_and_plot_with_map(sq_dumbbell, "fekete", test_degrees, target_deg=chosen_map_deg, res=150)

print("\n🧪 RUNNING EXPERIMENT: Discrete Leja Sequences on a Square Dumbbell Manifold")
make_table_and_plot_with_map(sq_dumbbell, "leja", test_degrees, target_deg=chosen_map_deg, res=150)

In [ ]:
rd_dumbbell = RoundDumbbell()

print("🧪 RUNNING EXPERIMENT: Approximate Fekete Points on a Round Dumbbell Manifold")
make_table_and_plot_with_map(rd_dumbbell, "fekete", test_degrees, target_deg=chosen_map_deg, res=200)

print("\n🧪 RUNNING EXPERIMENT: Discrete Leja Sequences on a Round Dumbbell Manifold")
make_table_and_plot_with_map(rd_dumbbell, "leja", test_degrees, target_deg=chosen_map_deg, res=200)

In [ ]:
annulus = Annulus()

print("🧪 RUNNING EXPERIMENT: Approximate Fekete Points on a Circular Annulus")
make_table_and_plot_with_map(annulus, "fekete", test_degrees, target_deg=chosen_map_deg, res=200)

print("\n🧪 RUNNING EXPERIMENT: Discrete Leja Sequences on a Circular Annulus")
make_table_and_plot_with_map(annulus, "leja", test_degrees, target_deg=chosen_map_deg, res=200)

In [ ]:
disjoint = DisjointSquares()

print("🧪 RUNNING EXPERIMENT: Approximate Fekete Points on Disjoint Square Domains")
make_table_and_plot_with_map(disjoint, "fekete", test_degrees, target_deg=chosen_map_deg, res=150)

print("\n🧪 RUNNING EXPERIMENT: Discrete Leja Sequences on Disjoint Square Domains")
make_table_and_plot_with_map(disjoint, "leja", test_degrees, target_deg=chosen_map_deg, res=150)

In [ ]:
right_triangle = Triangle()

print("🧪 RUNNING EXPERIMENT: Approximate Fekete Points on a Right Triangle")
make_table_and_plot_with_map(right_triangle, "fekete", test_degrees, target_deg=chosen_map_deg, res=150)

print("\n🧪 RUNNING EXPERIMENT: Discrete Leja Sequences on a Right Triangle")
make_table_and_plot_with_map(right_triangle, "leja", test_degrees, target_deg=chosen_map_deg, res=150)

In [ ]:
equi_triangle = EquilateralTriangle()

print("🧪 RUNNING EXPERIMENT: Approximate Fekete Points on an Equilateral Triangle")
make_table_and_plot_with_map(equi_triangle, "fekete", test_degrees, target_deg=chosen_map_deg, res=150)

print("\n🧪 RUNNING EXPERIMENT: Discrete Leja Sequences on an Equilateral Triangle")
make_table_and_plot_with_map(equi_triangle, "leja", test_degrees, target_deg=chosen_map_deg, res=150)

## Exporting Results

Run the following cell to archive all generated vector PDF plots into a single `.zip` package and automatically download it to your local machine.

In [ ]:
import glob
from google.colab import files

# Check if any PDF artifacts exist before archiving
pdf_files = glob.glob("*.pdf")

if pdf_files:
    print(f"Archiving {len(pdf_files)} PDF visualization charts...")
    # Zip all generated PDFs into a single clean archive
    !zip -q -r all_charts.zip *.pdf

    print("Initiating automatic download of 'all_charts.zip'...")
    files.download('all_charts.zip')
else:
    print("⚠️ No PDF files found in the workspace to archive. Run the experiment cells first.")